# Data Preparation — Solar Farm Location FYP

**Target variable**: `ALLSKY_SFC_SW_DWN` (Global Horizontal Irradiance, kWh/m²/day)

**Sources**: NASA POWER `SYN1DEG` + `MERRA2` daily Zarr datasets via AWS S3

**Regions**: Peninsular Malaysia, East Malaysia

---

### Phases
1. Library Imports
2. Data Loading from S3
3. Data Merging & Temporal Alignment
4. Feature Leakage Removal
5. Chronological Train / Val / Test Split
6. Feature Scaling
7. Temporal Feature Engineering
8. Outlier Handling
9. Save Processed Datasets

## Phase 1 — Library Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import s3fs
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

print("Libraries loaded successfully.")

## Phase 2 — Data Loading from NASA POWER S3

In [ ]:
fs = s3fs.S3FileSystem(anon=True)

SYN1DEG_DAILY = "s3://nasa-power/syn1deg/temporal/power_syn1deg_daily_temporal_utc.zarr"
MERRA2_DAILY  = "s3://nasa-power/merra2/temporal/power_merra2_daily_temporal_utc.zarr"

syn_ds   = xr.open_zarr(SYN1DEG_DAILY, storage_options={"anon": True})
merra_ds = xr.open_zarr(MERRA2_DAILY,  storage_options={"anon": True})

print("SYN1DEG time range:", syn_ds.time.values[0], "→", syn_ds.time.values[-1])
print("MERRA2  time range:", merra_ds.time.values[0], "→", merra_ds.time.values[-1])

In [ ]:
# Region bounding boxes
COUNTRY_BOUNDS = {
    "Peninsular Malaysia": {"lat_min": 1, "lat_max": 8,  "lon_min": 98,  "lon_max": 105},
    "East Malaysia":       {"lat_min": 1, "lat_max": 7,  "lon_min": 108, "lon_max": 120},
}

# Selected variables
SYN1DEG_VARS = ["ALLSKY_SFC_SW_DWN", "CLOUD_AMT", "AOD_55", "PW", "ALLSKY_KT", "PSH"]
MERRA2_VARS  = ["RH2M", "WS2M", "T2M_MAX", "PS"]

def select_region(ds, bounds):
    return ds.sel(
        lat=slice(bounds["lat_min"], bounds["lat_max"]),
        lon=slice(bounds["lon_min"], bounds["lon_max"])
    )

def extract_regional_df(ds, variables, bounds):
    """Extract variables for a region and return a daily DataFrame."""
    records = {}
    for var in variables:
        da_region = select_region(ds[var], bounds)
        records[var] = da_region.mean(dim=["lat", "lon"]).to_series()
    df = pd.DataFrame(records)
    df.index.name = "time"
    return df.dropna()

# Extract for all regions
syn_dfs   = {r: extract_regional_df(syn_ds,   SYN1DEG_VARS, b) for r, b in COUNTRY_BOUNDS.items()}
merra_dfs = {r: extract_regional_df(merra_ds, MERRA2_VARS,  b) for r, b in COUNTRY_BOUNDS.items()}

print("SYN1DEG  — Peninsular Malaysia shape:", syn_dfs["Peninsular Malaysia"].shape)
print("MERRA2   — Peninsular Malaysia shape:", merra_dfs["Peninsular Malaysia"].shape)

## Phase 3 — Data Merging & Temporal Alignment

SYN1DEG starts ~2000 while MERRA2 starts 1980. An **inner join** keeps only the overlapping date range.

In [ ]:
combined_dfs = {}

for region in COUNTRY_BOUNDS:
    merged = syn_dfs[region].join(merra_dfs[region], how="inner")
    combined_dfs[region] = merged
    print(f"{region}: {merged.shape[0]} days | {merged.index.min().date()} → {merged.index.max().date()}")

# Work with Peninsular Malaysia as the primary region
df = combined_dfs["Peninsular Malaysia"].copy()
df.head()

In [ ]:
# Verify no missing values after merge
print("Missing values after merge:")
print(df.isnull().sum())
print(f"\nTotal rows: {len(df)}")

## Phase 4 — Feature Leakage Removal

| Variable | Correlation with GHI | Action |
|---|---|---|
| `PSH` | 1.000 | ❌ Drop — derived directly from GHI |
| `ALLSKY_KT` | 0.978 | ❌ Drop — clearness index computed from GHI |
| All others | < 0.75 | ✅ Keep |

In [ ]:
TARGET = "ALLSKY_SFC_SW_DWN"
LEAKY  = ["PSH", "ALLSKY_KT"]

# Correlation check before removal
print("Correlations with target before leakage removal:")
print(df.corr()[TARGET].sort_values(ascending=False))

In [ ]:
# Separate features and target — drop leaky columns
X = df.drop(columns=[TARGET] + LEAKY)
y = df[TARGET]

print("Feature columns:", list(X.columns))
print("Target column  :", TARGET)
print("Feature shape  :", X.shape)

In [ ]:
# Correlation heatmap of cleaned features
fig, ax = plt.subplots(figsize=(8, 6))
corr_matrix = X.join(y).corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Feature Correlation Matrix (Leakage Removed)")
plt.tight_layout()
plt.show()

## Phase 5 — Chronological Train / Val / Test Split

> ⚠️ **Do NOT use random shuffling on time-series data.** Future data must never appear in the training set.

Split ratio: **70% train | 15% validation | 15% test**

In [ ]:
n         = len(X)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train, y_train = X.iloc[:train_end],       y.iloc[:train_end]
X_val,   y_val   = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test,  y_test  = X.iloc[val_end:],          y.iloc[val_end:]

print(f"Train : {X_train.index.min().date()} → {X_train.index.max().date()}  ({len(X_train):>5} days)")
print(f"Val   : {X_val.index.min().date()} → {X_val.index.max().date()}  ({len(X_val):>5} days)")
print(f"Test  : {X_test.index.min().date()} → {X_test.index.max().date()}  ({len(X_test):>5} days)")

In [ ]:
# Visual split timeline
fig, ax = plt.subplots(figsize=(12, 2))
ax.barh(0, len(X_train), left=0,          color="#4CAF50", label=f"Train ({len(X_train)}d)")
ax.barh(0, len(X_val),   left=train_end,  color="#FF9800", label=f"Val   ({len(X_val)}d)")
ax.barh(0, len(X_test),  left=val_end,    color="#F44336", label=f"Test  ({len(X_test)}d)")
ax.set_yticks([])
ax.set_xlabel("Day index")
ax.set_title("Chronological Train / Val / Test Split")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## Phase 6 — Feature Scaling

Apply **StandardScaler** (zero mean, unit variance). Fit **only** on training data.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   # Fit + transform on train
X_val_scaled   = scaler.transform(X_val)          # Transform only
X_test_scaled  = scaler.transform(X_test)         # Transform only

# Wrap back into DataFrames for readability
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_val_scaled   = pd.DataFrame(X_val_scaled,   columns=X.columns, index=X_val.index)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X.columns, index=X_test.index)

print("Scaled training data statistics:")
print(X_train_scaled.describe().round(3))

In [ ]:
# Save scaler
os.makedirs("../models", exist_ok=True)
joblib.dump(scaler, "../models/feature_scaler.pkl")
print("Scaler saved to ../models/feature_scaler.pkl")

## Phase 7 — Temporal Feature Engineering

Adding calendar-based and lag features to help the model capture seasonality and temporal dependencies.

In [ ]:
def add_temporal_features(df_features, df_target):
    """Add calendar and lag features. Returns (X_enriched, y_aligned)."""
    enriched = df_features.copy()

    # --- Calendar features ---
    doy = enriched.index.dayofyear
    enriched["doy_sin"]   = np.sin(2 * np.pi * doy / 365)
    enriched["doy_cos"]   = np.cos(2 * np.pi * doy / 365)
    month = enriched.index.month
    enriched["month_sin"] = np.sin(2 * np.pi * month / 12)
    enriched["month_cos"] = np.cos(2 * np.pi * month / 12)

    # --- Lag features (1, 3, 7 days) ---
    LAG_VARS = ["CLOUD_AMT", "RH2M", "AOD_55"]
    for var in LAG_VARS:
        if var in enriched.columns:
            for lag in [1, 3, 7]:
                enriched[f"{var}_lag{lag}"] = enriched[var].shift(lag)

    # --- Rolling statistics ---
    if "CLOUD_AMT" in enriched.columns:
        enriched["CLOUD_AMT_roll7"]  = enriched["CLOUD_AMT"].rolling(7).mean()
        enriched["CLOUD_AMT_roll30"] = enriched["CLOUD_AMT"].rolling(30).mean()

    # Drop NaN rows introduced by lags/rolling (keep target aligned)
    combined = enriched.join(df_target)
    combined = combined.dropna()
    return combined.drop(columns=[df_target.name]), combined[df_target.name]

# Apply to all splits
X_train_eng, y_train_eng = add_temporal_features(X_train, y_train)
X_val_eng,   y_val_eng   = add_temporal_features(X_val,   y_val)
X_test_eng,  y_test_eng  = add_temporal_features(X_test,  y_test)

print(f"Features before engineering : {X_train.shape[1]}")
print(f"Features after  engineering : {X_train_eng.shape[1]}")
print(f"\nNew feature columns: {[c for c in X_train_eng.columns if c not in X_train.columns]}")

In [ ]:
# Visualise cyclical day-of-year encoding
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(X_train_eng["doy_sin"], X_train_eng["doy_cos"],
                c=X_train_eng.index.dayofyear, cmap="plasma", s=5, alpha=0.7)
axes[0].set_title("Cyclical Day-of-Year Encoding")
axes[0].set_xlabel("doy_sin")
axes[0].set_ylabel("doy_cos")

axes[1].plot(y_train_eng.index, y_train_eng.values, lw=0.7, color="#FF6B35")
axes[1].set_title("GHI Time Series (Training Set)")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("ALLSKY_SFC_SW_DWN")

plt.tight_layout()
plt.show()

## Phase 8 — Outlier Handling

Clip extreme values to the 1st–99th percentile range computed from the training set. Bounds are derived from training data only to prevent leakage.

In [ ]:
def detect_outliers_iqr(series, factor=3.0):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    return (series < lower) | (series > upper)

print("Outlier count per feature (IQR × 3 rule, training set):")
outlier_summary = {}
for col in X_train_eng.columns:
    mask = detect_outliers_iqr(X_train_eng[col])
    count = mask.sum()
    outlier_summary[col] = count
    if count > 0:
        print(f"  {col:30s}: {count} outliers ({count/len(X_train_eng)*100:.2f}%)")

if sum(outlier_summary.values()) == 0:
    print("  No significant outliers detected.")

In [ ]:
# Clip to 1st–99th percentile bounds derived from training data
clip_bounds = {}
for col in X_train_eng.columns:
    low  = X_train_eng[col].quantile(0.01)
    high = X_train_eng[col].quantile(0.99)
    clip_bounds[col] = (low, high)

def apply_clip(df, bounds):
    df_clipped = df.copy()
    for col, (low, high) in bounds.items():
        if col in df_clipped.columns:
            df_clipped[col] = df_clipped[col].clip(low, high)
    return df_clipped

X_train_clean = apply_clip(X_train_eng, clip_bounds)
X_val_clean   = apply_clip(X_val_eng,   clip_bounds)
X_test_clean  = apply_clip(X_test_eng,  clip_bounds)

print("Clipping applied. Final feature shapes:")
print(f"  X_train : {X_train_clean.shape}")
print(f"  X_val   : {X_val_clean.shape}")
print(f"  X_test  : {X_test_clean.shape}")

## Phase 9 — Save Processed Datasets

In [ ]:
os.makedirs("../data/processed", exist_ok=True)

# Save feature matrices and targets
X_train_clean.to_parquet("../data/processed/X_train.parquet")
X_val_clean.to_parquet("../data/processed/X_val.parquet")
X_test_clean.to_parquet("../data/processed/X_test.parquet")

y_train_eng.to_frame().to_parquet("../data/processed/y_train.parquet")
y_val_eng.to_frame().to_parquet("../data/processed/y_val.parquet")
y_test_eng.to_frame().to_parquet("../data/processed/y_test.parquet")

# Save the full merged DataFrame for reference
for region, cdf in combined_dfs.items():
    fname = region.lower().replace(" ", "_")
    cdf.to_parquet(f"../data/processed/{fname}_combined.parquet")

print("✅ All processed datasets saved to ../data/processed/")

# Summary
print("\n--- Final Dataset Summary ---")
print(f"  Features          : {X_train_clean.shape[1]}")
print(f"  Training samples  : {len(X_train_clean)}")
print(f"  Validation samples: {len(X_val_clean)}")
print(f"  Test samples      : {len(X_test_clean)}")
print(f"  Target range      : {y_train_eng.min():.2f} – {y_train_eng.max():.2f} kWh/m²/day")